In [ ]:
import pandas as pd
from scipy.stats import fisher_exact

In [2]:
events_df = pd.read_csv('../../SV/SV_signature_analysis/Clustered_SVs/results/hotspot_event_definitions.csv')
events = (
    events_df
    .set_index('event')[['chrom', 'start', 'end']]
    .to_dict(orient='index')
)
# import file with gene coordinates
gene_allelic_file = pd.read_csv("../data/gene_coordinates.txt", sep="\t")

# Clean, map, and filter to 1-24
gene_allelic_file ['Chromosome/scaffold name'] = (
    gene_allelic_file['Chromosome/scaffold name']
    .replace({'X': '23', 'Y': '24'})
)

gene_allelic_file['Chromosome/scaffold name'] = pd.to_numeric(gene_allelic_file['Chromosome/scaffold name'])
gene_allelic_file = gene_allelic_file[(gene_allelic_file ['Chromosome/scaffold name'] >= 1) & (gene_allelic_file['Chromosome/scaffold name'] <= 24)]
gene_allelic_file['Chromosome/scaffold name'] = gene_allelic_file['Chromosome/scaffold name'].astype(int)
events

{'CCND3': {'chrom': 6, 'start': 39000000, 'end': 55000000},
 'MYC': {'chrom': 8, 'start': 99000000, 'end': 137000000},
 'CDK4-MDM2': {'chrom': 12, 'start': 53000000, 'end': 75000000},
 'CCNE1': {'chrom': 19, 'start': 27000000, 'end': 34000000},
 'TP53': {'chrom': 17, 'start': 7000000, 'end': 22000000}}

In [ ]:
for event, coords in events.items():
    print(f"\n{event} event")
    event = event.lower()
    sig_deseq_results = pd.read_csv(f'../outputs/output_files_{event}/sig_results_w_dfci.csv')
    deseq_results = pd.read_csv(f'../outputs/output_files_{event}/results_w_dfci.csv')
    sig_deseq_results_up = sig_deseq_results[sig_deseq_results['log2FoldChange'] > 0]
    print(f"Proportion of genes upregulated: {len(sig_deseq_results_up)/len(deseq_results)} ({len(sig_deseq_results_up)} / {len(deseq_results)})")

    # find the genes in the clustered SV region included in analysis
    in_region = gene_allelic_file[
        (gene_allelic_file['Chromosome/scaffold name'] == coords['chrom']) &
        (gene_allelic_file['Gene end (bp)']   >= coords['start']) &
        (gene_allelic_file['Gene start (bp)'] <= coords['end'])
    ]
    genes_in_region = sorted(set(in_region['Gene name']) & set(deseq_results['gene_name']))
    num_genes = len(genes_in_region)

    # find the proportion of these that are upregulated
    num_sig_genes = len(sig_deseq_results_up[sig_deseq_results_up['in_region'] == 'yes'])
    prop_sig_genes = num_sig_genes / num_genes
    print(f"Proportion of genes in region that are upregulated: {prop_sig_genes} ({num_sig_genes}/{num_genes})")

    # Fisher's exact test: locus vs background, excluding locus genes from background
    locus_gene_set = set(genes_in_region)
    up_gene_set    = set(sig_deseq_results_up['gene_name'])
    all_gene_set   = set(deseq_results['gene_name'])
    background_gene_set = all_gene_set - locus_gene_set

    a = len(locus_gene_set & up_gene_set)          # locus, upregulated
    b = len(locus_gene_set) - a                     # locus, not up
    c = len(background_gene_set & up_gene_set)       # background, upregulated
    d = len(background_gene_set) - c                 # background, not up

    odds_ratio, p_value = fisher_exact([[a, b], [c, d]], alternative='greater')
    bg_prop = c / (c + d)
    print(f"Background upregulated (locus excluded): {bg_prop:.4f} ({c}/{c + d})")
    print(f"Fold enrichment: {(a / (a + b)) / bg_prop:.1f}x")
    print(f"Fisher's exact (one-sided): OR={odds_ratio:.2f}, p={p_value:.2e}")


CCND3 event
Proportion of genes upregulated: 0.003213592541919883 (101 / 31429)
Proportion of genes in region that are upregulated: 0.1895424836601307 (29/153)
Background upregulated (locus excluded): 0.0023 (72/31276)
Fold enrichment: 82.3x
Fisher's exact (one-sided): OR=101.36, p=6.74e-44

MYC event
Proportion of genes upregulated: 0.008113525724649209 (255 / 31429)
Proportion of genes in region that are upregulated: 0.14492753623188406 (20/138)
Background upregulated (locus excluded): 0.0075 (235/31291)
Fold enrichment: 19.3x
Fisher's exact (one-sided): OR=22.40, p=1.87e-19

CDK4-MDM2 event
Proportion of genes upregulated: 0.011072576283050686 (348 / 31429)
Proportion of genes in region that are upregulated: 0.14655172413793102 (34/232)
Background upregulated (locus excluded): 0.0100 (313/31197)
Fold enrichment: 14.6x
Fisher's exact (one-sided): OR=16.94, p=5.71e-28

CCNE1 event
Proportion of genes upregulated: 0.010181679340736263 (320 / 31429)
Proportion of genes in region that a